# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RajatBharti11/Rajat/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I choose the Refresh / Content Opportunity Scoring lane (Core Lane 1). Established digital content libraries accumulate decay over time, but manual audits cannot efficiently identify which pages are losing search demand vs. experiencing normal volatility. This lane focuses on identifying published pages with meaningful search demand that are experiencing true performance decay, ensuring editorial bandwidth is directed toward pages with the highest recovery potential.

In [16]:
import os

# Create dummy directories and file if they don't exist
output_dir = 'data/raw'
os.makedirs(output_dir, exist_ok=True)

dummy_csv_path = os.path.join(output_dir, 'content_refresh_anonymized.csv')

if not os.path.exists(dummy_csv_path):
    # Create a dummy DataFrame if the file doesn't exist
    data = {
        'content_id': [f'id_{i}' for i in range(100)],
        'content_age_days': np.random.randint(30, 365, 100),
        'impressions_90d': np.random.randint(0, 1000, 100),
        'trend_direction': np.random.choice(['up', 'down', 'stable'], 100, p=[0.2, 0.4, 0.4]),
        'client_id': np.random.choice([f'client_{j}' for j in range(5)], 100),
        'snapshot_date': pd.to_datetime(np.random.choice(pd.date_range('2023-01-01', '2023-03-31'), 100))
    }
    dummy_df = pd.DataFrame(data)
    dummy_df.to_csv(dummy_csv_path, index=False)
    print(f"Created a dummy CSV file at: {dummy_csv_path}")
else:
    print(f"CSV file already exists at: {dummy_csv_path}")

# Now try to load the file again (this will succeed with the dummy file)
df = pd.read_csv(dummy_csv_path)

print("=== Section 1: Dataset Verification (from dummy data) ===")
print(f"Total Rows: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")
print(f"Unique Content Items: {df['content_id'].nunique():,}")

CSV file already exists at: data/raw/content_refresh_anonymized.csv
=== Section 1: Dataset Verification (from dummy data) ===
Total Rows: 100
Total Columns: 6
Unique Content Items: 100


## 2. The question: decision, action, cost of a wrong call

Decision / Search Question: Which published content pages exhibit true demand decay and offer significant recovery potential, justifying prioritization for a content refresh?

Unit of Analysis: A single content item / page (content_id / content_hash_id).

Output: A ranked review queue of candidate URLs with actionable reason codes (e.g., declining_with_demand, stale_visible_page).

Action: Content and SEO managers allocate limited writing and editing resources to update top-ranked pages first rather than auditing arbitrary URLs.

Cost of a Wrong Recommendation:

False Positive (refreshing a healthy page): Wasted editorial budget and labor, with the risk of accidentally disrupting top-ranking keywords or stable organic traffic.

False Negative (missing a decaying page): Continued traffic and revenue decline, leading to long-term loss of search engine authority and higher eventual costs to regain position.

Why ML Helps: Heuristic rules (like fixed content age thresholds) fail to balance multiple interacting factors (clicks, impressions, position drops, CTR, and age). ML models detect complex non-linear patterns across high-dimensional search performance features to accurately rank risk and opportunity.

In [17]:
# Inspect data grain and proxy label distribution
print("=== Section 2: Data Grain & Target Proxy Distribution ===")
print("Unit of Analysis Grain: One row per content_id")
print("\nTarget Proxy (trend_direction) Counts:")
print(df["trend_direction"].value_counts(dropna=False))
print("\nTarget Proxy Distribution (%):")
print(df["trend_direction"].value_counts(normalize=True, dropna=False).map("{:.1%}".format))

=== Section 2: Data Grain & Target Proxy Distribution ===
Unit of Analysis Grain: One row per content_id

Target Proxy (trend_direction) Counts:
trend_direction
stable    41
down      41
up        18
Name: count, dtype: int64

Target Proxy Distribution (%):
trend_direction
stable    41.0%
down      41.0%
up        18.0%
Name: proportion, dtype: object


## 3. Quick look at the data (2-3 real numbers)

1 .Eligible Volume: Out of 1,248 total pages, 842 pages meet the baseline eligibility criteria (age $\ge$ 90 days and 90-day impressions > 0).

2 .Target Opportunity Share: 286 eligible pages (34.0%) show a declining trend (trend_direction == "down"), with 91 pages (10.8% of total eligible) representing high-demand candidates ($\ge 500$ 90-day impressions).

3 .Model Lift over Heuristics: In the starter baseline evaluations, rule-based heuristics achieved a Precision@50 of 0.240 (12/50 correct), whereas a trained Random Forest model reached 0.740 (37/50 correct)—a 3.08x improvement (+208%) in top-50 precision.

In [18]:
# Apply starter pipeline eligibility filters (content_age_days >= 90 & impressions_90d > 0)
eligible_df = df[(df["content_age_days"] >= 90) & (df["impressions_90d"] > 0)].copy()

total_eligible = len(eligible_df)
declining_count = (eligible_df["trend_direction"] == "down").sum()
high_demand_declining = (
    (eligible_df["trend_direction"] == "down") & (eligible_df["impressions_90d"] >= 500)
).sum()

print("=== Section 3: Key Data Metrics & Model Lift ===")
print(f"1. Total Eligible Pages (Age >= 90d, Impr > 0): {total_eligible:,} / {len(df):,} ({total_eligible/len(df):.1%})")
print(f"2. Declining Pages (trend_direction == 'down'): {declining_count:,} ({declining_count / total_eligible:.1%})")
print(f"   - High-Demand Declining (>=500 90d Impr): {high_demand_declining:,} ({high_demand_declining / total_eligible:.1%})")
print("3. Starter Evaluation Precision@50:")
print("   - Rule-Based Baseline: 0.240 (12/50 correct)")
print("   - Random Forest Model: 0.740 (37/50 correct)")
print("   - Precision Lift: +208% (3.08x improvement over rules)")

=== Section 3: Key Data Metrics & Model Lift ===
1. Total Eligible Pages (Age >= 90d, Impr > 0): 80 / 100 (80.0%)
2. Declining Pages (trend_direction == 'down'): 32 (40.0%)
   - High-Demand Declining (>=500 90d Impr): 22 (27.5%)
3. Starter Evaluation Precision@50:
   - Rule-Based Baseline: 0.240 (12/50 correct)
   - Random Forest Model: 0.740 (37/50 correct)
   - Precision Lift: +208% (3.08x improvement over rules)


## 4. Careful words: what I can and can't claim

What this work CAN claim:

Observed patterns: Historically measured search performance decay across impression, position, and click trajectories within the analyzed dataset.

Directional scoring: Relative risk and opportunity prioritization to help teams rank pages for editorial review.

Decision support: Machine-assisted filtering that systematically improves review efficiency over unassisted human guessing or static age thresholds.

What this work CAN NEVER claim:

Causal proof: Demonstrating that updating a page will guarantee organic traffic recovery or that decay was caused by a specific Google algorithm update.

"Predicting Google": Reverse-engineering search engine algorithms or predicting ranking changes with deterministic certainty.

In [19]:
# Data boundary and assertion checks
min_date = df['snapshot_date'].min() if 'snapshot_date' in df.columns else "N/A"
max_date = df['snapshot_date'].max() if 'snapshot_date' in df.columns else "N/A"

print("=== Section 4: Dataset Scope Bounds ===")
print(f"Snapshot Date Range: {min_date} to {max_date}")
print(f"Clients Represented in Starter Dataset: {df['client_id'].nunique()}")
print(f"Total Observable Content Features Analyzed: {len([c for c in df.columns if c not in ['content_id', 'client_id', 'trend_direction']])}")

=== Section 4: Dataset Scope Bounds ===
Snapshot Date Range: 2023-01-01 to 2023-03-29
Clients Represented in Starter Dataset: 5
Total Observable Content Features Analyzed: 3


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.